# 01 - Data Loading and Inspection
## Credit Risk Analysis — Give Me Some Credit

**Objetivo de este notebook:**
- Cargar el dataset original `cs-training.csv`
- Realizar una primera inspección de los datos
- Entender qué significa cada variable
- Identificar los primeros problemas de calidad de datos

**Dataset:** Give Me Some Credit — Kaggle (150.000 registros, 12 variables)

In [1]:
# Importamos las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
# Cargamos el dataset
df = pd.read_csv('../data/raw/cs-training.csv')
df.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


# Observaciones a primera vista del Dataset:
- Hay 12 columnas.
- 'Unnamed' podría ser un índice duplicado, no aporta nada, se puede eliminar la columna entera
- 'SerioousDlquin2yrs' puede ser la variable objetivo. En "DATA DICTIONARY.XLS" dice que es la columna que representa si la persona experimentó 90 días de morosidad o peor, comprendiendo que 1 es "sí" y 2 es "no".
- En 'MonthlyIncome' que contiene los ingresos mensuales, y 'NumberOfDependents' que contiene el nº de personas en la unidad familiar excluyendo a la persona que tiene el crédito, los dos tienen nº decimales, así que puede que haya valores nulos que hay que revisar.

In [7]:
df.shape

(150000, 12)

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 11  NumberOfDep

# Observaciones:
- Según el "Data Dictionary", las columnas 'MonthlyIncome' y 'NumberOfDependents' deberían de ser integer y aquí podemos observar que son float.
- En 'MonthlyIncome hay 120.269 de 150.000 datos disponibles, es decir, hay datos nulos que hay que tratar (29.731 valores no disponibles)
- Lo mismo de antes ocurre con 'NumberOfDependents' con 146.076 de 150.000 datos (3.924 valores no disponibles)
- Estoy más segura que Unnamed era el índice en el .csv y pandas la ha leido como una columna más

In [9]:
df.describe()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
count,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,150000.000000,1.202690e+05,150000.000000,150000.000000,150000.000000,150000.000000,146076.000000
mean,75000.500000,0.066840,6.048438,52.295207,0.421033,353.005076,6.670221e+03,8.452760,0.265973,1.018240,0.240387,0.757222
std,43301.414527,0.249746,249.755371,14.771866,4.192781,2037.818523,1.438467e+04,5.145951,4.169304,1.129771,4.155179,1.115086
min,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000
25%,37500.750000,0.000000,0.029867,41.000000,0.000000,0.175074,3.400000e+03,5.000000,0.000000,0.000000,0.000000,0.000000
50%,75000.500000,0.000000,0.154181,52.000000,0.000000,0.366508,5.400000e+03,8.000000,0.000000,1.000000,0.000000,0.000000
75%,112500.250000,0.000000,0.559046,63.000000,0.000000,0.868254,8.249000e+03,11.000000,0.000000,2.000000,0.000000,1.000000
max,150000.000000,1.000000,50708.000000,109.000000,98.000000,329664.000000,3.008750e+06,58.000000,98.000000,54.000000,98.000000,20.000000


# Observaciones:

- 'age' el mínimo es 0 y máximo 109 son valores imposibles para un titular de una línea de crédito. Serán Outliers que están pendientes de tratar.
- 'RevolvingUtilizationOfUnsecuredLines' es un porcentaje, debería estar entre 0 y 1. Su máximo de 50.708 es un outlier extremo porque es imposible en la realidad.
- 'DebtRatio'su máximo de 329.664 es imposible (deuda = 329.664 veces los ingresos). Outlier extremo.
- 'NumberOfTime30-59DaysPastDueNotWorse', 'NumberOfTimes90DaysLate' y 'NumberOfTime60-89DaysPastDueNotWorse' — los tres tienen máximo 98, valor imposible en 2 años.Probablemente estén realcionados, estás columnas nos dicen el nº de días que han pasado ha tener un retrasado en 30-59 días, 60-89 dias y 90 días el pago pero sin que haya empeorado la situación en los últimos 2 años, asi que el valor máximo debería de ser 24. Quizás sea una forma de dar valor a datos nulos
- 'MonthlyIncome' — count de 120.269 confirma 29.731 nulos.
- 'NumberOfDependents' — count de 146.076 confirma 3.924 nulos.